* This is an update to Project 1, dubbed "Salty Winter". I'm correcting the data analysis & visualization
(Lede Program)

Here's the original Github page to the project: 

https://github.com/Liyanziqia/project_salty_winter

Here're the major data discrepencies after verifying the data with the research team at NYC Department of Sanitation who got back to me with the correct data and analysis a few weeks after the project was submitted:

When cleaning the data, I assigned January–March to the previous calendar year, and left April in the current calendar year. This caused late-spring storms (such as an April 4, 2018 storm) to be grouped to the 2018–2019 season instead of the 2017–2018 season.  The Correction: DSNY defines a winter season as running continuous from autumn through spring, which makes sense in real life. An April storm belongs to the season that began the previous fall. 

In addition: the visualization concerpt is problematic too:
My initial assumption for visualization was that DSNY trucks carry around 22 tons of salt per load.  The research team at DSNY said they measure truck loads by volume (cubic yards), not weight.


Below're the original research question and workflow: 

# Research question: We had a harsh winter. Did New York City use the highest amount of road salt to melt snow over the past winter? 

# Goal: focusing on getting and cleaning the data I need, even if visualization is not beautiful.

# Due date: June 28th, 2026; 1st update: August 3rd, 2026

# Overall workflow: Use API to get data, then save raw CSV (archive), clean DataFrame, then analysis, then charts/visual. It's a simple project by design so keeping everything in one notebook.

  1. Getting data: fetch data from NYC Open Data, look at them, and save each as a CSV.
  2. Cleaning data: practice API, pandas, chart tools
  3. Visualize data: scale visualization(some sort of visual to showcase SCALE). 
        If time allows, I'll also show one reservoir location on map to show salt-related chemical increase in water.
*Document the learning process for future reference because I'll soon forget about everything.

Dataset: one single dataset (keep it simple)
Department of Sanitation (DSNY) Salt Usage
source: https://data.cityofnewyork.us/City-Government/DSNY-Salt-Usage/tavr-zknk/about_data
*note: I had oringially thought about other datasets, such as DEP Drinking Water Quality Distribution Monitoring, but scrapped that idea to avoid overthinking.

API workflow:

Build the URL
Make the request with requests.get()
Convert JSON to Python data with df
Create a DataFrame
Inspect the results

---
## how NYC Open Data's API works (what kind of API I'm dealing with)
# The website took me to formal documentation about Socrata APIs
https://dev.socrata.com/docs/queries/
# some how-to tutorials and resources:
this one explains Socrata API Basics:
https://github.com/mebauer/sodapy-tutorial-nyc-opendata/blob/main/socrata-api-basics.ipynb

# data summary:
NYC Open Data runs on a platform called Socrata (something new to me). Every dataset has its own URL that ends in '.json'. that's the API endpoint. e.g. https://data.cityofnewyork.us/resource/DATASET-ID.json

The dataset ID, unlike Pokeman API, is that  8-character code in the URL of any dataset page (e.g. 'tavr-zknk'). Can find it by querying the dataset page on NYC Open Data and clicking "API" in the top right , or just looking at the URL. 

Department of Sanitation (DSNY) Salt Usage
 https://data.cityofnewyork.us/City-Government/DSNY-Salt-Usage/tavr-zknk/about_data
 The intro page actually shows what's in the dataframe, which is helpful. I do want to somehow get the "Total Tons
 The total tons of salt DSNY dispensed onto roadways."
The plan now is: to get the total tons data for five boroughs for the LATEST winter season; then zoom in the top borough (Queens? and get historical data just for that borough)

#Poking around on the website
--found a specific documentation for this dataset
https://dev.socrata.com/foundry/data.cityofnewyork.us/tavr-zknk
--API end point
https://data.cityofnewyork.us/api/v3/views/tavr-zknk/query.json?

In [3]:
# Setting up tools, base_URL.
# Trying to get the data with API 
# Import the tools needed:

import requests
import pandas as pd

BASE_URL = "https://data.cityofnewyork.us/resource/"

print("Base URL ready.")


Base URL ready.


In [ ]:
#   1. Getting data
# Fetching the salt data
# Dataset ID: tavr-zknk
# Build the URL manually

salt_url = BASE_URL + "tavr-zknk.json"

print("Requesting:", salt_url)

salt_response = requests.get(salt_url)

print("Status code:", salt_response.status_code)

salt_data = salt_response.json()

print("Rows returned:", len(salt_data))

df_salt = pd.DataFrame(salt_data)

print("DataFrame created!!!")

df_salt.to_csv("df_salt_raw.csv", index=False)

print("Raw data saved.")

Requesting: https://data.cityofnewyork.us/resource/tavr-zknk.json
Status code: 200
Rows returned: 224
DataFrame created!!!
Raw data saved.


In [3]:
#checking raw data
df_salt.head()

,dsny_storm,date_of_report,manhattan,bronx,brooklyn,queens,staten_island,total_tons
0,Storm 1,2016-01-19T00:00:00.000,1111,2059,2690,6625,2931,15416
1,Storm 2,2016-01-23T00:00:00.000,2919,4567,5521,9253,4264,26524
2,Storm 2,2016-01-24T00:00:00.000,6582,8167,12290,17133,5411,49583
3,Storm 2,2016-01-25T00:00:00.000,3862,4094,6217,10236,1256,25665
4,Storm 2,2016-01-26T00:00:00.000,2499,3299,5120,7685,764,19367


In [4]:
df_salt.columns

Index(['dsny_storm', 'date_of_report', 'manhattan', 'bronx', 'brooklyn',
       'queens', 'staten_island', 'total_tons'],
      dtype='str')

In [5]:
df_salt.shape

(224, 8)

In [6]:
df_salt.tail()

,dsny_storm,date_of_report,manhattan,bronx,brooklyn,queens,staten_island,total_tons
219,Storm 4,2026-01-17T00:00:00.000,2734,3253,3392,8678,1730,19787
220,Storm 5,2026-01-18T00:00:00.000,8517,5322,11710,5698,8647,39894
221,Storm 6,2026-01-25T00:00:00.000,22666,20449,34078,48369,21401,146963
222,Storm 7,2026-02-15T00:00:00.000,3210,2240,4653,5888,3341,19332
223,Storm 8/9,2026-02-22T00:00:00.000,21355,17811,36323,41657,16612,133758


*** HERE'S WHERE I GOT THE DATA WRONG:
  2. Cleaning data: 

#OK. It's a ten-year span showing salt use by each storm. I need to figure out:
How much salt was used during the most recent winter?

I need to: ADD APRIL TO THE MONTHS when defining winter season

Convert the date column to a datetime.
Create a winter-season label/column.
Sum all storms within each winter.

In [5]:
df_salt['date_of_report'] = pd.to_datetime(df_salt['date_of_report'])
df_salt['year'] = df_salt['date_of_report'].dt.year
df_salt['month'] = df_salt['date_of_report'].dt.month

In [6]:
df_salt['winter_start'] = df_salt['year']
df_salt.loc[
    df_salt['month'].isin([1, 2, 3, 4]), # ADDED April here
    'winter_start'
] = df_salt['year'] - 1

df_salt['winter_season'] = (
    df_salt['winter_start'].astype(str)
    + "-"
    + (df_salt['winter_start'] + 1).astype(str).str[-2:]
)

In [8]:
df_salt.head()

,dsny_storm,date_of_report,manhattan,bronx,brooklyn,queens,staten_island,total_tons,year,month,winter_start,winter_season
0,Storm 1,2016-01-19,1111,2059,2690,6625,2931,15416,2016,1,2015,2015-16
1,Storm 2,2016-01-23,2919,4567,5521,9253,4264,26524,2016,1,2015,2015-16
2,Storm 2,2016-01-24,6582,8167,12290,17133,5411,49583,2016,1,2015,2015-16
3,Storm 2,2016-01-25,3862,4094,6217,10236,1256,25665,2016,1,2015,2015-16
4,Storm 2,2016-01-26,2499,3299,5120,7685,764,19367,2016,1,2015,2015-16


In [9]:
df_salt.to_csv("df_salt_winter.csv", index=False)

print("winter data saved.")

winter data saved.


In [10]:
#checking recent winter data
df_salt.tail()    

,dsny_storm,date_of_report,manhattan,bronx,brooklyn,queens,staten_island,total_tons,year,month,winter_start,winter_season
219,Storm 4,2026-01-17,2734,3253,3392,8678,1730,19787,2026,1,2025,2025-26
220,Storm 5,2026-01-18,8517,5322,11710,5698,8647,39894,2026,1,2025,2025-26
221,Storm 6,2026-01-25,22666,20449,34078,48369,21401,146963,2026,1,2025,2025-26
222,Storm 7,2026-02-15,3210,2240,4653,5888,3341,19332,2026,2,2025,2025-26
223,Storm 8/9,2026-02-22,21355,17811,36323,41657,16612,133758,2026,2,2025,2025-26


#see which season has the highest use of salt: group by 'winter_season', then sort 
*** ran the code below and found that datatype the 'total_tons' is string not a number ***
??? is there a code to see the datatype of all the columns?
# df_season_salt = df_salt.groupby('winter_season')['total_tons'].sum().reset_index().sort_values(by='total_tons', ascending=False)

In [11]:
# retry: need to convert the total_tons column into actual mathematical integers or floats before runninggroupby().
# fix the data type, recalculate the sum, and sort the results properly:
# 1. Force the total_tons column to be numeric (errors='coerce' turns bad data like text into NaN)
df_salt['total_tons'] = pd.to_numeric(df_salt['total_tons'], errors='coerce')

# 2. Fill any missing rows with 0 just in case
df_salt['total_tons'] = df_salt['total_tons'].fillna(0)

# 3. Now run your group and sum code!
df_season_salt = df_salt.groupby('winter_season')['total_tons'].sum().reset_index().sort_values(by='total_tons', ascending=False)

# 4. View the real mathematical results
df_season_salt.head(10)


,winter_season,total_tons
10,2025-26,504811
5,2020-21,453763
2,2017-18,447060
1,2016-17,363191
3,2018-19,354400
6,2021-22,337890
0,2015-16,266265
9,2024-25,234845
4,2019-20,203502
8,2023-24,195757


# Double checking the corrected data agasint the original data provided by Department of Sanitation (DSNY) research team, who said via email:
"Your summary of salt usage is mostly correct, with one exception: The 2017-2018 salt usage should be 447,060 tons and the 2018-2019 salt usage should be 354,400 tons. The tonnage from 4/4/18 from OpenData (13,666 tons) was added to 2018-2019 instead of 2017-2018. Other than that, your total salt usage numbers from the summary align with DSNY’s records." 
# data correct this time!!

In [13]:
#save the df sorted by total_tons of each winter_season into a csv
df_season_salt.to_csv("df_salt_winter_sorted by total tons.csv", index=False)

print("Sorted data saved.")

Sorted data saved.


#   3. Visualize data: 
# I think I'm ready to make two charts. 
# I would like to chart with Altair, which I just learned it this morning. how should I do that? I wanted to create two charts with df_season_salt and df_season_borough, to show city-wide, total tons of last winter was the highest, and by borough, for last winter season, queens was the highest, respectively


In [16]:
#quitely install altair

%pip install -q altair vega_datasets


import altair as alt

#building city-wide chart, but keeping it them vertical with flat labels

# Build the city-wide historical chart
chart_city_wide_flat = alt.Chart(df_season_salt).mark_bar().encode(
    # 'labelAngle=0' forces the winter season text to stay perfectly horizontal
    x=alt.X('winter_season:N', title='Winter Season', axis=alt.Axis(labelAngle=0)),
    # 'total_tons:Q' plots the numeric quantity on the vertical axis
    y=alt.Y('total_tons:Q', title='Total Tons of Salt Used'),
    # Adds interactive hover tooltips for digital news layouts
    tooltip=['winter_season', 'total_tons']
).properties(
    title='NYC used nearly more than half a million tons of salt last winter season, highest in the last 10 years',
    width=600,
    height=400
)

# Display the final chart
chart_city_wide_flat




Note: you may need to restart the kernel to use updated packages.


alt.Chart(...)

Technical Context for visualization: 
Salt Weight vs. Volume: DSNY measures truck loads by volume in cubic yards, not weight in tons.  Rock Salt Density: Standard highway rock salt weighs approximately 1.08 tons per cubic yard.

# Standard bulk rock salt density: ~80 lbs/cu. ft (~2,160 lbs or 1.08 short tons per cubic yard)
# Note: Bulk salt density typically ranges from 0.95 to 1.08 tons/cu. yd depending on moisture and settling.
TONS_PER_CU_YD = 1.08


In [22]:
#   5: VISUALIZE TRUCK LOADS IN ALTAIR -- correct conversion from tons to truck loads

# ---   1: CONVERSION PARAMETERS ---
TONS_PER_CU_YD = 1.08
SPREADER_CAPACITY_CU_YD = 16  # "most DSNY salt spreaders hold around 16 cubic yards of salt"

# ---   2: CALCULATE CUBIC YARDS & SPREADER LOADS ---
df_salt['total_cu_yards'] = df_salt['total_tons'] / TONS_PER_CU_YD
df_salt['spreader_loads'] = df_salt['total_cu_yards'] / SPREADER_CAPACITY_CU_YD

# ---   3: CREATE THE MISSING DATAFRAME (df_seasonal_loads) ---
df_seasonal_loads = df_salt.groupby('winter_season').agg(
    total_tons=('total_tons', 'sum'),
    total_cu_yards=('total_cu_yards', 'sum'),
    spreader_loads=('spreader_loads', 'sum')
).reset_index()

# Clean up numbers for plotting
df_seasonal_loads['total_cu_yards'] = df_seasonal_loads['total_cu_yards'].round(0)
df_seasonal_loads['spreader_loads'] = df_seasonal_loads['spreader_loads'].round(0)

# Sort by season chronologically or by load size
df_seasonal_loads = df_seasonal_loads.sort_values(by='winter_season')

# Quick check to confirm it exists now
df_seasonal_loads.tail()

,winter_season,total_tons,total_cu_yards,spreader_loads
6,2021-22,337890,312861.0,19554.0
7,2022-23,80720,74741.0,4671.0
8,2023-24,195757,181256.0,11329.0
9,2024-25,234845,217449.0,13591.0
10,2025-26,504811,467418.0,29214.0


In [21]:
#save the the seasonal loads into a csv
df_seasonal_loads.to_csv("df_salt_winter_seasonal_loads.csv", index=False)


# calculations for the pictogram visual:
## 
To correct previous volume assumptions, all truck load calculations follow standard operational parameters provided by the **NYC Department of Sanitation (DSNY) research team**:


| Winter Season | Total Salt Dispensed | Total Salt Volume  | Equivalent Spreader Loads | Icon Count (1 icon truck  = 500  Trucks ) |
| :--- | :--- | :--- | :--- | :--- |
| **2015–16** | 266,265  | 246,542   | **14,942 truckloads** | **30 icons** (rounded from 29) |
| **2020–21** | 453,763  | 420,151   | **25,464 truckloads** | **51 icons** (rounded from 50.93) |
| **2025–26** | 504,811  | 467,418   | **28,328 truckloads** | **57 icons** (rounded from 56.66) |

---

> **Caption / Legend Copy:**  
> `each icon = 500 trucks · avg. 16.5 cu. yards per spreader truck (1.08 tons/cu. yd)

## Visual before:
https://liyanziqia.github.io/project_salty_winter/
## Visual after: 
https://liyanziqia.github.io/project_salty_winter_1stUpdate/